# Suite1 Project6 Ethics

**Dataset:** California Housing

---
Run each cell in order. All outputs are generated from real data.

In [ ]:
# Setup: imports and configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plotting style
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams['figure.dpi'] = 120

# Helper: load dataset
def load_data(name):
    return pd.read_csv(f'https://raw.githubusercontent.com/ulink-ibcs/A4_ML_Projects/main/datasets/{name}')

print("✅ Setup complete!")

In [ ]:
"""Project 6: Ethics & Bias Audit"""
import sys, os; sys.path.insert(0, os.path.dirname(__file__))

df = load_csv('housing.csv')
OUTPUT_DIR = os.path.join(BASE_DIR, 'suite1_project6_outputs'); os.makedirs(OUTPUT_DIR, exist_ok=True)

## PROJECT 6: ETHICS & BIAS AUDIT — Examining Model Fairness

In [ ]:
X = df[['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']]
y = df['MedHouseVal']

# ── 1. Simulate a Sensitive Attribute ──

## 1. Defining Sensitive Attributes

In [ ]:
# Use Latitude as proxy for "region" (north/south divide)
lat_median = df['Latitude'].median()
df['Region'] = np.where(df['Latitude'] >= lat_median, 'North', 'South')
df['IncomeGroup'] = pd.qcut(df['MedInc'], q=3, labels=['Low Income', 'Middle Income', 'High Income'])

print(f"Region split: {df['Region'].value_counts().to_dict()}")
print(f"Income group split:\n{df['IncomeGroup'].value_counts().to_string()}")

# ── 2. Train Model ──

## 2. Train Base Model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
overall_r2 = r2_score(y_test, y_pred)
overall_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Overall Model: R²={overall_r2:.4f}, RMSE={overall_rmse:.4f}")

# ── 3. Bias Detection by Region ──

## 3. Bias Detection: North vs South Regions

In [ ]:
test_df = X_test.copy()
test_df['Actual'] = y_test
test_df['Predicted'] = y_pred
test_df['Region'] = df.loc[test_df.index, 'Region'].values
test_df['IncomeGroup'] = df.loc[test_df.index, 'IncomeGroup'].values
test_df['Error'] = test_df['Predicted'] - test_df['Actual']
test_df['AbsError'] = np.abs(test_df['Error'])

region_metrics = test_df.groupby('Region').agg({
    'Actual': 'mean', 'Predicted': 'mean', 'Error': 'mean', 'AbsError': 'mean',
    'Actual': ['mean', 'std', 'count']
}).round(4)
print("Model Performance by Region:")
print(region_metrics.to_string())

# Bias metrics
print("\n=== Per-Region Error Analysis ===")
for region in ['North', 'South']:
    subset = test_df[test_df['Region'] == region]
    rmse_r = np.sqrt(mean_squared_error(subset['Actual'], subset['Predicted']))
    r2_r = r2_score(subset['Actual'], subset['Predicted'])
    mean_err = subset['Error'].mean()
    print(f"  {region:6s}: RMSE={rmse_r:.4f}, R²={r2_r:.4f}, Mean Error={mean_err:.4f}")

# Demographic Parity difference
print(f"\n=== Fairness Metric: Demographic Parity ===")
pred_rates = test_df.groupby('Region')['Predicted'].mean()
print(f"Mean predicted price by region:")
print(pred_rates.to_string())
print(f"Difference: ${abs(pred_rates['North'] - pred_rates['South'])*100:.1f}K")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=test_df, x='Region', y='Error', ax=axes[0], palette='Set2')
axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[0].set_title('Prediction Error by Region', fontsize=13)
axes[0].set_ylabel('Error = Predicted - Actual ($100K)')
axes[0].grid(True, alpha=0.3)

sns.boxplot(data=test_df, x='IncomeGroup', y='Error', ax=axes[1], palette='Set2', 
            order=['Low Income', 'Middle Income', 'High Income'])
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[1].set_title('Prediction Error by Income Group', fontsize=13)
axes[1].set_ylabel('Error = Predicted - Actual ($100K)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
# 'p6_bias_boxplots')
plt.show()
print("[Saved] p6_bias_boxplots.png")

# ── 4. Environmental Impact ──

## 4. Environmental Impact Assessment

In [ ]:
# Estimate training energy
import time

print("=== Energy Consumption Estimates ===")
print(f"Dataset size: {len(df):,} rows × {X.shape[1]} features")

# Training time benchmark
start = time.time()
_ = GridSearchCV(Ridge(), {'alpha': [0.01, 0.1, 1, 10]}, cv=3).fit(X.values[:1000], y.values[:1000])
small_time = time.time() - start

start = time.time()
_ = GridSearchCV(Ridge(), {'alpha': [0.01, 0.1, 1, 10]}, cv=3).fit(X.values, y.values)
full_time = time.time() - start

print(f"Small grid (1000 samples): {small_time:.2f}s")
print(f"Full grid (20K samples):   {full_time:.2f}s")
print(f"Scale factor: {full_time/small_time:.1f}x for {len(df)/1000:.0f}x more data")

# Carbon estimate (rough)
print(f"\nEstimated energy for full training:")
print(f"  CPU time: {full_time:.1f}s")
print(f"  Est. power draw: ~65W (laptop CPU)")
print(f"  Est. energy: {full_time * 65 / 3600:.4f} kWh")
print(f"  Est. CO₂ (avg grid): {full_time * 65 / 3600 * 0.5:.4f} kg CO₂")

print(f"\nFor neural network training (GPU, hours):")
print(f"  Est. energy: ~300W × 24h = 7.2 kWh")
print(f"  Est. CO₂: ~3.6 kg CO₂")

# ── 5. Privacy & Consent ──

## 5. Privacy & Consent Considerations

In [ ]:
print("""
=== Key Ethical Questions for Discussion ===

1. BIAS: The model systematically underprices northern regions.
   → Is this a fair prediction or algorithmic bias?
   → What if this model is used for property tax assessment?

2. PRIVACY: The data includes precise locations (lat/lon).
   → What could be inferred about individuals from this data?
   → Is aggregated census data sufficient for model training?

3. ACCOUNTABILITY: Who is responsible if the model makes a biased decision?
   → The developer? The data scientist? The institution?
   → How transparent should the model's decision process be?

4. ENVIRONMENTAL: Each experiment consumes energy.
   → How many experiments before the carbon cost matters?
   → Should students reduce grid search granularity to save energy?

5. SOCIETAL IMPACT: Automated property valuation affects housing affordability.
   → Could this model be used to discriminate against certain neighbourhoods?
   → What guardrails would you propose?
""")

results_final = {
    'overall_r2': float(overall_r2),
    'overall_rmse': float(overall_rmse),
    'north_south_gap': float(abs(pred_rates['North'] - pred_rates['South'])),
    'region_rmse': {'North': float(np.sqrt(mean_squared_error(test_df[test_df['Region']=='North']['Actual'], 
                                                               test_df[test_df['Region']=='North']['Predicted']))),
                    'South': float(np.sqrt(mean_squared_error(test_df[test_df['Region']=='South']['Actual'], 
                                                               test_df[test_df['Region']=='South']['Predicted'])))},
    'training_time_full': float(full_time),
    'est_co2_kg': float(full_time * 65 / 3600 * 0.5),
}
    json.dump(results_final, f, indent=2, default=str)

print("Done.")